# Safety-Incident Anomaly Pattern Detection

**INT396 · SIH26165**  
This notebook is part of a reproducible project comparing Isolation Forest and Local Outlier Factor on OSHA Severe Injury Reports.

**Interpretation rule:** model flags are statistical outliers, not verified anomalies, accident predictions, or proof of unsafe conduct.

Select the project's Python environment as the notebook kernel. The raw dataset must be available at `data/raw/January2015toNovember2025.csv`.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
from safety_anomaly.data.io import load_incidents
from safety_anomaly.data.audit import audit_frame
from safety_anomaly.visualization.plots import save_missingness_figure
import pandas as pd
from IPython.display import display
DATA_PATH = ROOT / 'data/raw/January2015toNovember2025.csv'
assert DATA_PATH.exists(), f'Missing dataset: {DATA_PATH}'
print('Dataset:', DATA_PATH)
print('Size (MB):', round(DATA_PATH.stat().st_size / 1024**2, 2))

## 2. Load and inspect the dataset
The audit uses the supplied raw file without modifying it.

In [ ]:
df = load_incidents(DATA_PATH)
print('Shape:', df.shape)
safe_preview_cols = [c for c in ['EventDate', 'Primary NAICS', 'EventTitle', 'NatureTitle', 'Part of Body Title', 'SourceTitle'] if c in df]
display(df[safe_preview_cols].head(5))
display(pd.DataFrame({'column': df.columns, 'dtype': df.dtypes.astype(str).values}))

## 3. Data quality summary

In [ ]:
audit = audit_frame(df)
summary_keys = ['rows', 'columns', 'exact_duplicate_rows', 'unique_upa',
                'missing_upa', 'duplicated_id_rows_including_first',
                'id_distinct_values_among_repeated', 'event_date_min',
                'event_date_max', 'invalid_event_dates', 'empty_narratives']
display(pd.DataFrame({'metric': summary_keys,
                      'value': [audit.get(k) for k in summary_keys]}))
missingness = pd.DataFrame(audit['missingness'])
display(missingness.head(20))

## 4. Missingness visualization

In [ ]:
fig_path = ROOT / 'reports/figures/missingness_top_columns.png'
save_missingness_figure(df, fig_path)
from IPython.display import Image, display
display(Image(filename=str(fig_path)))

## 5. Date and identifier validation

In [ ]:
dates = pd.to_datetime(df['EventDate'], errors='coerce')
print('Date minimum:', dates.min())
print('Date maximum:', dates.max())
print('Invalid dates:', int(dates.isna().sum()))
print('UPA unique:', int(df['UPA'].nunique()))
print('Rows with repeated ID values:', int(df['ID'].duplicated(keep=False).sum()))
print('Exact duplicate rows:', int(df.duplicated().sum()))
assert df['UPA'].notna().all()
assert df['UPA'].is_unique, 'UPA uniqueness check failed; review before modeling.'
assert df.duplicated().sum() == 0, 'Exact duplicates exist; inspect before any removal.'

## 6. Save audit artifacts

In [ ]:
out = ROOT / 'reports/data_audit'
out.mkdir(parents=True, exist_ok=True)
(out / 'audit_summary.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
missingness.to_csv(out / 'missingness.csv', index=False)
print('Saved audit outputs to', out)

## Interpretation notes
Record observations about missing fields, repeated source IDs, reporting coverage and invalid values here. Do not treat the repeated `ID` values as duplicates when `UPA` is unique and full rows differ.